# Advanced Retrieval: Filtering Documents with Similarity Score Thresholds

Retrieval-Augmented Generation (RAG) systems are powerful, but their performance is fundamentally limited by the quality of the context they receive. If a retriever pulls in irrelevant or low-quality documents, even the most advanced Large Language Model (LLM) will struggle to generate an accurate answer—a phenomenon often called "garbage in, garbage out." While standard retrieval methods aim for the *top K* results regardless of score, real-world data is noisy. Some retrieved documents might be technically related but semantically weak, diluting the context and confusing the final generation step.

This notebook introduces a critical advanced technique: **similarity score thresholding**. Instead of blindly accepting the top $K$ documents, we learn to set a minimum acceptable similarity score ($\tau$). By implementing this filter, we instruct the retriever to discard any document whose calculated cosine similarity falls below $\tau$. This mechanism acts as an intelligent quality gate, ensuring that only highly relevant and strongly correlated context is passed downstream.

Mastering threshold-based retrieval is crucial for building robust, production-grade RAG pipelines, especially when integrating them into complex state machines like LangGraph. By controlling the input context's quality at the retrieval step, developers can significantly improve answer fidelity, reduce hallucinations caused by noise, and build more reliable agents that operate with surgical precision.

***

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Understand Retrieval Limitations:** Articulate why simply retrieving the top $K$ documents is insufficient for high-fidelity RAG systems.
*   **Implement Score Filtering:** Utilize `similarity_score_threshold` in a vector store retriever to filter results based on a minimum similarity score ($\tau$).
*   **Analyze Relevance Scores:** Interpret and utilize cosine similarity scores (and their inverse, $1 - \text{cosine\_score}$) to gauge the semantic relevance of retrieved documents.
*   **Improve Context Quality:** Apply advanced retrieval techniques to ensure that only highly relevant context is passed to the generation step, thereby improving overall system reliability.


### Setup and Environment Loading

This cell initializes the environment by loading necessary credentials (like `OPENAI_API_KEY`) from a `.env` file. It imports core libraries for embedding generation (`OpenAIEmbeddings`), vector storage (`Chroma`), document handling (`Document`), and unique ID generation (`uuid4`).


In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from uuid import uuid4


# Load OPENAI_API_KEY from .env file to ensure credentials are available for API calls
load_dotenv()


True

### Document Setup (The Corpus)

This cell initializes a list of `Document` objects, simulating the retrieved knowledge base or corpus. It is crucial for demonstrating advanced RAG techniques because it provides a controlled mix of highly relevant ('ml'), somewhat relevant ('economics'), and completely off-topic documents ('cooking', 'sports') to test the effectiveness of similarity scoring thresholds.


In [2]:
# 12 documents: ML (highly relevant to query), economics (somewhat relevant), cooking and sports (off-topic)
# The threshold filter will progressively cut out the lower-scoring documents
docs = [
    Document(page_content="Supervised learning trains models on labeled input-output pairs to predict unseen data.", metadata={"topic": "ml"}),
    Document(page_content="Neural networks learn by adjusting weights through backpropagation and gradient descent.", metadata={"topic": "ml"}),
    Document(page_content="Overfitting occurs when a model memorises training data and performs poorly on new data.", metadata={"topic": "ml"}),
    Document(page_content="Training data quality and quantity are the most important factors in model performance.", metadata={"topic": "ml"}),
    Document(page_content="Cross-validation splits data into folds to evaluate model generalisation more reliably.", metadata={"topic": "ml"}),
    Document(page_content="Inflation is the rate at which the general level of prices for goods and services rises over time.", metadata={"topic": "economics"}),
    Document(page_content="GDP measures the total monetary value of all goods and services produced within a country.", metadata={"topic": "economics"}),
    Document(page_content="Interest rates set by central banks influence borrowing costs and consumer spending.", metadata={"topic": "economics"}),
    Document(page_content="Caramelisation occurs when sugar is heated above 160°C, creating complex flavour compounds.", metadata={"topic": "cooking"}),
    Document(page_content="Fermentation uses microorganisms to convert sugars into alcohol or acids, preserving food.", metadata={"topic": "cooking"}),
    Document(page_content="A marathon is a long-distance race of exactly 42.195 kilometres, run on roads.", metadata={"topic": "sports"}),
    Document(page_content="Tennis scoring follows a love, 15, 30, 40, game sequence, with deuce at 40-40.", metadata={"topic": "sports"}),
]

### Vectorstore Initialization

This cell initializes the ChromaDB vector store. It uses `OpenAIEmbeddings` to generate embeddings for text and configures the underlying HNSW index with cosine similarity, which is crucial for accurate semantic search in RAG systems.


In [3]:
# Embed documents and store in ChromaDB
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    embedding_function=embeddings,
    collection_name="demo",
    collection_configuration={
        "hnsw": {
            "space": "cosine"      # cosine, L2, ip
        }
    }
)


### Document Ingestion into Vector Store

This cell is responsible for ingesting the processed documents (`docs`) into the configured vector store. It uses `vectorstore.add_documents()` to perform the embedding and storage, ensuring each document receives a unique identifier generated by `uuid4()`.


In [4]:
# add documents to vector store

# The add_documents method takes the list of processed documents (docs).
# It also requires a corresponding list of unique IDs for each document.
ids_added = vectorstore.add_documents(documents=docs,
                                      ids=[str(uuid4()) for _ in range(len(docs))])


In [5]:
print(len(ids_added))

12


### Code Explanation

This cell simply calculates and displays the total number of documents (or chunks) stored in the `docs` variable. This is crucial for verifying that the document loading or retrieval process successfully retrieved the expected amount of data before applying any filtering or scoring logic.


In [6]:
len(docs) # Calculates the length of the 'docs' list/variable, which represents the total number of documents (chunks) retrieved from the vector store.


12

### Code Explanation

This cell performs a similarity search against the vector store, retrieving the top $k$ documents (here, $k=3$) along with their calculated similarity scores. This is crucial for evaluating the relevance of retrieved context before passing it to an LLM, allowing us to implement thresholding logic.


In [9]:
# retrieve top 5 documents with score
query = "How does ML model training work?"

# Perform a similarity search and retrieve the top k=3 documents along with their scores.
similarity_scores = vectorstore.similarity_search_with_score(query, k=3)

# Iterate through the retrieved documents and print the score, topic, and content for inspection.
for doc, score in similarity_scores:
    print(f"Score: {score:.4f} | Topic: {doc.metadata['topic']} | Content: {doc.page_content}")


Score: 0.5683 | Topic: ml | Content: Training data quality and quantity are the most important factors in model performance.
Score: 0.5720 | Topic: ml | Content: Supervised learning trains models on labeled input-output pairs to predict unseen data.
Score: 0.6079 | Topic: ml | Content: Overfitting occurs when a model memorises training data and performs poorly on new data.


### Code Explanation

This cell iterates through the stored similarity scores, which are assumed to be in a (1-score) format. It calculates and prints the actual cosine similarity score by subtracting the stored value from 1, allowing for easier interpretation of the relevance metric.


In [10]:
# get all the scores in (1-score) format
# actual cosine similarity scores
for _, score in similarity_scores:
    # Calculate the true cosine similarity score (1 - stored_value)
    print(f"Similarity Score: {1 - score:.4f}")


Similarity Score: 0.4317
Similarity Score: 0.4280
Similarity Score: 0.3921


### Code Explanation

This cell demonstrates how to filter retrieved documents using a minimum similarity score threshold. By setting `search_type="similarity_score_threshold"` and providing a `score_threshold`, the retriever only returns documents whose cosine similarity to the query exceeds the specified value, ensuring relevance.


In [13]:
query = "How does ML model training work?"

threshold = 0.43

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": threshold},
)

results = retriever.invoke(query)

print(f"=== Threshold = {threshold} — {len(results)} document(s) returned ===")
for i, doc in enumerate(results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")


=== Threshold = 0.43 — 1 document(s) returned ===
  [1] topic=ml: Training data quality and quantity are the most important factors in model performance.


### Code Explanation

This cell simply displays the contents of the `results` variable, which is expected to be a pandas DataFrame or similar structured object containing the final similarity scores and associated metadata from the previous steps. This allows for immediate inspection and validation of the computed results.


In [8]:
results


[]